<a href="https://colab.research.google.com/github/lucmos2002/workshopGITHUB/blob/master/03x_attention_Lucas_Souza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mecanismos de Atenção

## 1 Atenção causal e bidirecional

Durante a aula você aprendeu sobre o mencanismo de atenção usado nos modelos de linguagem atuais. Criamos a classe `SelfAttention_v2` com atenção bidirecional, onde cada token pode dar atenção tanto para os tokens que vem antes na sentença, como pode dar atenção para os tokens subsequentes. Em seguida, adicionamos causalidade na atenção e criamos a classe `CausalAttention`, que faz com que os tokens dêem atenção apenas para os tokens passados, comumente usada nos grandes modelos de linguagem generativos.
<br><br>
Porém, modelos como o T5 usam a arquitetura enconder-decoder, que mistura os dois tipos de atenção: bidirecional (nos enconders) e causal (nos decoders). Tendo isso em mente, faça o seguinte:

- Baseando-se na classe `CausalAttention`, crie a classe `SelfAttention_v3` que poderá se comportar tanto com atenção bidirecional quanto com atenção causal. Para isso, adicione o parâmetro booleano `is_causal` no construtor da classe, que aplicará o filtro causal na matriz de atenção caso o parâmetro seja verdadeiro e bidirecional caso seja falso. Altere também o retorno do método forward para retornar tanto o `context_vec`quanto o `attn_weights`.

- Em seguida passe o input definido abaixo pela pela nova classe `SelfAttention_v3` usando a arquitetura bidirecional e também a arquitetura causal e imprima o resultado dos vetores de contexto e dos pesos de atenção nas duas situações para comparar a diferença entre eles.

In [3]:
import torch
import torch.nn as nn

inputs = torch.tensor(
  [[[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]]] # step     (x^6)
)

d_in, d_out = 3, 2
context_length=6
dropout=0.0

In [18]:
class SelfAttention_v3(nn.Module):
    def __init__(self, d_in, d_out, dropout, context_length, qkv_bias=False, is_causal=True): # COMPLETE OS PARÂMETROS
        super().__init__()
        # SEU CÓDIGO
        self.is_causal = is_causal # Store is_causal as an instance attribute

        if is_causal:
          self.d_out = d_out
          self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
          self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
          self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
          self.dropout = nn.Dropout(dropout) # New
          self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

        else:
          self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
          self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
          self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x): # COMPLETE OS PARÂMETROS
        # SEU CÓDIGO
        if self.is_causal:
          b, num_tokens, d_in = x.shape
          keys = self.W_key(x)
          queries = self.W_query(x)
          values = self.W_value(x)

          attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
          attn_scores.masked_fill_(  # New, _ ops are in-place
              self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
          attn_weights = torch.softmax(
              attn_scores / keys.shape[-1]**0.5, dim=-1
          )
          attn_weights = self.dropout(attn_weights) # New

          context_vec = attn_weights @ values
          return context_vec, attn_weights

        else:
          keys = self.W_key(x)
          queries = self.W_query(x)
          values = self.W_value(x)

          attn_scores = queries @ keys.transpose(1, 2)
          attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

          context_vec = attn_weights @ values
          return context_vec, attn_weights

In [19]:
# PASSE O INPUT PELA SelfAttention_v3 COM O PARÂMETRO is_causal=True
# E IMPRIMA A MATRIZ DE ATENÇÃO RETORNADA

sa_v3 = SelfAttention_v3(d_in, d_out, dropout, context_length,is_causal=True)
context_vec, attn_weights = sa_v3(inputs)
print(attn_weights)

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5398, 0.4602, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3698, 0.3150, 0.3152, 0.0000, 0.0000, 0.0000],
         [0.2601, 0.2408, 0.2406, 0.2584, 0.0000, 0.0000],
         [0.2157, 0.1892, 0.1895, 0.2007, 0.2049, 0.0000],
         [0.1717, 0.1593, 0.1591, 0.1730, 0.1623, 0.1746]]],
       grad_fn=<SoftmaxBackward0>)


In [20]:
# PASSE O INPUT PELA SelfAttention_v3 COM O PARÂMETRO is_causal=false
# E IMPRIMA A MATRIZ DE ATENÇÃO RETORNADA
sa_v3 = SelfAttention_v3(d_in, d_out, dropout, context_length,is_causal=False)
context_vec, attn_weights = sa_v3(inputs)
print(attn_weights)

tensor([[[0.1714, 0.1606, 0.1611, 0.1679, 0.1773, 0.1618],
         [0.1605, 0.1683, 0.1683, 0.1679, 0.1667, 0.1683],
         [0.1604, 0.1682, 0.1682, 0.1680, 0.1671, 0.1682],
         [0.1623, 0.1691, 0.1689, 0.1669, 0.1640, 0.1687],
         [0.1603, 0.1652, 0.1655, 0.1694, 0.1736, 0.1660],
         [0.1631, 0.1704, 0.1701, 0.1660, 0.1607, 0.1697]]],
       grad_fn=<SoftmaxBackward0>)


## 2 Multi-head Attention + Data Loading

Use o data loader visto nas aulas anteriores para tokenizar e processar o texto abaixo para a tarefa de previsão do próximo token e passe o primeiro batch de dados pela camada de multi-head attention visto na aula passada e imprima o shape da saída. Lembre-se que os tokens gerados pelo data loader devem passar pela camada de token embeddings e position embeddings (token_embedding + position_embedding) antes de passar pela atenção. Essas camadas também já foram definidas abaixo.

In [21]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


time_machine_text = "The Time Traveller (for so it will be convenient to speak of him) \
was expounding a recondite matter to us. His grey eyes shone and \
twinkled, and his usually pale face was flushed and animated. The \
fire burned brightly, and the soft radiance of the incandescent \
lights in the lilies of silver caught the bubbles that flashed and \
passed in our glasses. Our chairs, being his patents, embraced and \
caressed us rather than submitted to be sat upon, and there was that \
luxurious after-dinner atmosphere when thought roams gracefully \
free of the trammels of precision. And he put it to us in this \
way--marking the points with a lean forefinger--as we sat and lazily \
admired his earnestness over this new paradox (as we thought it) \
and his fecundity."


vocab_size = 50257
emb_dim = 256
context_length = 1024


token_embedding_layer = nn.Embedding(vocab_size, emb_dim)
pos_embedding_layer = nn.Embedding(context_length, emb_dim)

In [23]:
# DEFINA A CLASSE DE DATASET, O DATA LOADER E A CLASSE DE MULTI HEAD ATTENTION
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text as a single sequence of IDs
        token_ids = tokenizer.encode(txt)

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [22]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [24]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [25]:
# CARREGUE O DATA LOADER COM O TEXTO ACIMA
# PASSE O PRIMEIRO BATCH PELA CAMADA DE EMBEDDING
# PASSE O EMBEDDING PELA MULTI HEAD ATTENTION
# IMPRIMA O SHAPE DA SAÍDA

max_length = 4
batch_size = 8
stride = 4
context_length = 4
num_heads = 2
d_in = emb_dim
d_out = emb_dim


In [26]:
dataloader = create_dataloader_v1(time_machine_text, batch_size=8, max_length=4, stride=4, shuffle=True)

In [28]:
# Obtém apenas o primeiro batch
inputs, targets = next(iter(dataloader))

print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)
print("\nInputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs shape: torch.Size([8, 4])
Targets shape: torch.Size([8, 4])

Inputs:
 tensor([[  649, 22226,   357,   292],
        [ 1986,   373, 44869,   290],
        [ 4105, 11542,  2759,  1479],
        [  284,   514,   287,   428],
        [13791,  2951, 44193,   290],
        [  674, 15232,    13,  3954],
        [  356,  1807,   340,     8],
        [ 1424,   286, 15440,    13]])

Targets:
 tensor([[22226,   357,   292,   356],
        [  373, 44869,   290, 15108],
        [11542,  2759,  1479,   286],
        [  514,   287,   428,   835],
        [ 2951, 44193,   290,   665],
        [15232,    13,  3954, 18791],
        [ 1807,   340,     8,   290],
        [  286, 15440,    13,   843]])


In [29]:
# Criar posições para a sequência (0 a 3, pois context_length = 4)
pos = torch.arange(0, inputs.shape[1], device=inputs.device)

# Somar Token Embeddings + Position Embeddings
x = token_embedding_layer(inputs) + pos_embedding_layer(pos)

In [30]:
# Instanciar a camada
mha = MultiHeadAttention(
    d_in=emb_dim,
    d_out=emb_dim,
    context_length=context_length,
    dropout=0.0,
    num_heads=num_heads
)

# Passar a representação combinada
out = mha(x)

# Imprimir o shape final
print("Shape da saída:", out.shape)

Shape da saída: torch.Size([8, 4, 256])
